# Sample Backtest — BTC RSI + EMA Crossover
Downloads data, runs a backtest, saves results to TimescaleDB, plots equity curve.

In [ ]:
# Cell 1 — Download data
import os
import datetime
os.environ['ARCHIVE_PATH'] = '/data/lake/bronze'

from fast_trade.archive.update_kline import update_kline

# 2 weeks only — fast download (~30 seconds)
start = datetime.datetime(2025, 2, 1)
end   = datetime.datetime(2025, 2, 14)

print('Downloading... this takes ~30 seconds')
update_kline('BTCUSDT', 'binanceus', start_date=start, end_date=end)
print('Download complete')

In [ ]:
# Cell 2 — Define strategy and run backtest
from fast_trade import run_backtest

strategy = {
    'freq': '4H',
    'symbol': 'BTCUSDT',
    'exchange': 'binanceus',
    'start': '2025-02-01',
    'stop': '2025-02-14',
    'base_balance': 10000.0,
    'comission': 0.001,
    'datapoints': [
        {'name': 'rsi',       'transformer': 'rsi', 'args': [14]},
        {'name': 'ema_fast',  'transformer': 'ema', 'args': [9]},
        {'name': 'ema_slow',  'transformer': 'ema', 'args': [21]},
    ],
    'enter': [
        ['rsi', '<', 45],
        ['ema_fast', '>', 'ema_slow'],
    ],
    'exit': [
        ['rsi', '>', 65],
    ],
}

result = run_backtest(strategy)
summary = result['summary']
print('Backtest complete')

In [ ]:
# Cell 3 — Print summary
import pprint
pprint.pprint(summary)

In [ ]:
# Cell 4 — Save to TimescaleDB (visible in Grafana + Dashboard)
import os, uuid, json
from datetime import datetime, timezone
from sqlalchemy import create_engine, text

engine = create_engine(os.environ['DATABASE_URL'])
run_id = str(uuid.uuid4())
strategy_name = 'BTC_RSI_EMA_Crossover'

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO strategies (name, version, config)
        VALUES (:name, 1, CAST(:config AS jsonb))
        ON CONFLICT DO NOTHING
    """), {'name': strategy_name, 'config': json.dumps(strategy)})

    strategy_id = conn.execute(text(
        'SELECT id FROM strategies WHERE name = :name ORDER BY version DESC LIMIT 1'
    ), {'name': strategy_name}).scalar()

    conn.execute(text("""
        INSERT INTO backtest_runs
            (id, strategy_id, strategy_hash, data_hash, status, started_at, finished_at, summary)
        VALUES
            (:id, :sid, :shash, :dhash, 'done', :started, :finished, CAST(:summary AS jsonb))
    """), {
        'id': run_id,
        'sid': strategy_id,
        'shash': str(hash(json.dumps(strategy, sort_keys=True))),
        'dhash': 'manual',
        'started': datetime.now(timezone.utc),
        'finished': datetime.now(timezone.utc),
        'summary': json.dumps(summary, default=str),
    })

    # Insert trade log into trades table so Grafana panels populate
    trade_df = result['trade_df'].copy()
    if not trade_df.empty:
        trade_df.index = pd.to_datetime(trade_df.index)
        rows = []
        for ts, row in trade_df.iterrows():
            action = 'enter' if row.get('in_trade', 0) == 1 else 'exit'
            rows.append({
                'ts': ts,
                'run_id': run_id,
                'portfolio_name': strategy_name,
                'symbol': strategy['symbol'],
                'exchange': strategy.get('exchange', ''),
                'action': action,
                'price': float(row.get('close', 0)),
                'qty': 0.0,
                'pnl_perc': float(row.get('adj_account_value_change_perc', 0) or 0),
                'pnl_abs': float(row.get('adj_account_value_change', 0) or 0),
                'hold_bars': None,
            })
        conn.execute(text("""
            INSERT INTO trades
                (ts, run_id, portfolio_name, symbol, exchange, action, price, qty, pnl_perc, pnl_abs, hold_bars)
            VALUES
                (:ts, :run_id, :portfolio_name, :symbol, :exchange, :action, :price, :qty, :pnl_perc, :pnl_abs, :hold_bars)
        """), rows)

print(f'Saved run: {run_id}')
print(f"Return : {summary.get('return_perc')}%")
print(f"Sharpe : {summary.get('sharpe_ratio')}")
print(f"Trades : {summary.get('num_trades')}")

In [ ]:
# Cell 5 — Plot equity curve
import plotly.graph_objects as go

df = result['df']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df['adj_account_value'],
    name='Strategy', line=dict(color='royalblue', width=2)
))
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['close'] / df['close'].iloc[0] * strategy['base_balance'],
    name='Buy & Hold', line=dict(color='gray', width=1, dash='dash')
))
fig.update_layout(
    title='BTC RSI+EMA Crossover — Equity Curve',
    yaxis_title='USD',
    xaxis_title='Date',
    height=500,
    hovermode='x unified'
)
fig.show()

In [ ]:
# Cell 6 — Trade log
result['trade_df']